# 02 — scikit-learn classification training

Trains a `RandomForestClassifier` (in a `Pipeline` with `StandardScaler` + `OneHotEncoder`) on the
AML data asset registered by `01a-copy-fabric-to-aml.ipynb`.

- **Local smoke test** (cell 2): runs `src.train.train(...)` against the local parquet snapshot.
- **Remote AML job** (cells 4-5): submits a `command` job to `cpu-cluster` that consumes the
  `azureml:contoso-poc-dataset:1` MLTable.
- **Register model** (cell 7): registers the resulting MLflow model.

Per the 6 May 2026 sync: **scikit-learn only**, no AutoML, no deep learning.

In [ ]:
import os, sys, pathlib
os.environ["PYTHONUTF8"] = "1"
# Force UTF-8 default encoding for open() on Windows
if sys.platform == "win32":
    import _locale
    _locale._getdefaultlocale = lambda *args: ('en_US', 'UTF-8')

%load_ext autoreload
%autoreload 2

# Ensure project root is on sys.path regardless of kernel cwd
_project_root = str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from dotenv import load_dotenv
load_dotenv(pathlib.Path(_project_root) / ".env", encoding="utf-8", override=True)

DATA_ASSET = 'contoso-poc-dataset'   # registered by 01a-copy-fabric-to-aml
DATA_VERSION = '1'
TARGET = 'isPaidTimeOff'
DROP_COLS = ['countryRegionCode']     # leaky vs countryOrRegion
COMPUTE = 'cpu-cluster'
MODEL_NAME = 'contoso-poc-model'

## 1. Local smoke test

Runs the full training pipeline against the local parquet snapshot for fast iteration.

In [ ]:
from src.train import train

metrics = train(
    data_path='../data/local/publicholidays_clf.parquet',
    target=TARGET,
    drop_cols=DROP_COLS,
    output_dir='../outputs/model',
)
metrics

## 2. Submit remote training job to `cpu-cluster`

The job:
- Mounts `azureml:contoso-poc-dataset:1` MLTable as `${{inputs.training_data}}`.
- Runs `python -m src.train` inside the curated `sklearn-1.5` environment.
- MLflow autolog captures parameters, metrics, and system info; the trained model is saved
  to `outputs/model/` so AML keeps it as a job artifact.

In [ ]:
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.identity import DefaultAzureCredential
from src.config import load_settings

s = load_settings()
ml = MLClient(DefaultAzureCredential(), s.subscription_id, s.resource_group, s.aml_workspace)

job = command(
    code='..',  # repo root, so `python -m src.train` resolves
    command=(
        'python -m src.train '
        '--data-path ${{inputs.training_data}} '
        f'--target {TARGET} '
        f'--drop-cols {" ".join(DROP_COLS)} '
        '--output-dir outputs/model'
    ),
    inputs={
        'training_data': Input(
            type=AssetTypes.MLTABLE,
            path=f'azureml:{DATA_ASSET}:{DATA_VERSION}',
            mode=InputOutputModes.RO_MOUNT,
        ),
    },
    environment='azureml://registries/azureml/environments/sklearn-1.5/labels/latest',
    compute=COMPUTE,
    display_name='contoso-poc-sklearn-training',
    experiment_name='contoso-poc',
)
returned_job = ml.jobs.create_or_update(job)
print(f'Job: {returned_job.name}')
print(f'Studio: {returned_job.studio_url}')

In [ ]:
ml.jobs.stream(returned_job.name)
final = ml.jobs.get(returned_job.name)
print('Status:', final.status)

## 3. Register the trained MLflow model from the job

In [ ]:
from azure.ai.ml.entities import Model

model = Model(
    name=MODEL_NAME,
    path=f'azureml://jobs/{returned_job.name}/outputs/artifacts/paths/outputs/model/',
    type=AssetTypes.MLFLOW_MODEL,
    description='scikit-learn isPaidTimeOff classifier trained on Fabric publicholidays.',
)
registered = ml.models.create_or_update(model)
print(f'Registered: {registered.name} v{registered.version}')